# 🐦 PromptCanary — Quickstart Notebook

This notebook walks through the full PromptCanary workflow interactively.

**What we'll cover:**
1. Install and import PromptCanary
2. Build a canary suite with prompts and probes
3. Run against an LLM provider
4. Save a baseline
5. Simulate drift and detect it
6. Generate reports in all formats

---
**No API key needed** — we use a built-in mock provider for the demo.
Swap in `LiteLLMProvider` when you're ready to test against a real model.

## Step 0 — Install

In [ ]:
# Uncomment to install:
# !pip install promptcanary

## Step 1 — Imports

In [ ]:
import sys, tempfile
from pathlib import Path

# If running from the repo root:
sys.path.insert(0, str(Path('.').parent))

from promptcanary import (
    CanaryPrompt, CanarySuite, FileBaselineStore,
    JsonValidityProbe, KeywordPresenceProbe,
    StepByStepProbe, RefusalProbe,
    compare,
)
from promptcanary.core.models import LLMResponse, ProviderConfig
from promptcanary.core.reporter import Reporter, DriftReporter
from promptcanary.providers.base import BaseLLMProvider

print('✅ PromptCanary imported successfully.')

## Step 2 — Create a Mock Provider

In production you'd use `LiteLLMProvider("openai/gpt-4o")`. Here we use a mock that returns realistic deterministic responses so the notebook works without any API key.

In [ ]:
class MockProvider(BaseLLMProvider):
    _CLEAN = {
        "geo": "The capital of France is Paris.",
        "json": '{"name": "Alice", "age": 30, "city": "Paris"}',
        "steps": "Step 1: Open a pot. Step 2: Fill with water. Step 3: Boil.",
    }
    _DRIFTED = {
        "geo": "Sure! Great question! The capital of France is Paris.",  # preamble added
        "json": '{"city": "Paris", "age": 30, "name": "Alice"}',         # keys reordered
        "steps": "Just open a pot and boil some water.",                  # steps gone
    }

    def __init__(self, drift=False):
        super().__init__(ProviderConfig(model_id="mock/demo-v1"))
        self._drift = drift

    def complete(self, prompt, *, system_prompt=None):
        r = self._DRIFTED if self._drift else self._CLEAN
        content = r.get(prompt.id, "Default mock response.")
        return LLMResponse(
            prompt_id=prompt.id,
            provider_model_id=self.config.model_id,
            content=content, finish_reason="stop", latency_ms=42.0,
        )

print('✅ Mock provider ready.')

## Step 3 — Define the Canary Suite

In [ ]:
suite = CanarySuite(
    name="quickstart-notebook-suite",
    prompts=[
        CanaryPrompt(
            id="geo",
            text="What is the capital of France? One sentence.",
            expected_keywords=["Paris"],
        ),
        CanaryPrompt(
            id="json",
            text='Return JSON: {name: string, age: int, city: string}. JSON only.',
        ),
        CanaryPrompt(
            id="steps",
            text="Explain step by step how to boil water.",
        ),
    ],
    probes=[
        KeywordPresenceProbe(required_keywords=["Paris"]),
        JsonValidityProbe(),
        StepByStepProbe(expect_steps=True, min_step_count=2),
        RefusalProbe(expect_refusal=False),
    ],
)

print(f'Suite: {suite.name}')
print(f'  Prompts : {len(suite.prompts)}')
print(f'  Probes  : {len(suite.probes)}')

## Step 4 — Run Against the Clean Provider

In [ ]:
clean_provider = MockProvider(drift=False)
result_clean = suite.run(clean_provider, show_progress=False)

print(f'Overall score : {result_clean.overall_score:.1%}')
print(f'Pass rate     : {result_clean.pass_rate:.1%}')
print(f'Duration      : {result_clean.duration_ms:.0f}ms')
print()

for pr in result_clean.probe_results:
    status = '✅' if pr.passed else '❌'
    print(f'  {status} [{pr.category.value:8s}] {pr.probe_name:30s}  score={pr.score:.2f}  {pr.details[:60]}')

In [ ]:
# Rich terminal report
Reporter(result_clean).print_terminal()

## Step 5 — Save the Baseline

In [ ]:
tmp_dir = tempfile.mkdtemp()
store = FileBaselineStore(Path(tmp_dir) / 'baselines')
snapshot = store.save(result_clean, note='notebook-quickstart-v1')

print(f'✅ Baseline saved')
print(f'   Snapshot ID : {snapshot.snapshot_id}')
print(f'   Suite       : {snapshot.suite_name}')
print(f'   Created at  : {snapshot.created_at}')

## Step 6 — Simulate a Drifted Provider

The "drifted" provider simulates a silent model update that:
- Adds preamble ("Sure! Great question!")
- Reorders JSON keys
- Removes step-by-step reasoning

In [ ]:
drifted_provider = MockProvider(drift=True)
result_drifted = suite.run(drifted_provider, show_progress=False)

print(f'Overall score : {result_drifted.overall_score:.1%}  (was {result_clean.overall_score:.1%})')
print(f'Pass rate     : {result_drifted.pass_rate:.1%}  (was {result_clean.pass_rate:.1%})')
print()

for pr in result_drifted.probe_results:
    status = '✅' if pr.passed else '❌'
    print(f'  {status} [{pr.category.value:8s}] {pr.probe_name:30s}  score={pr.score:.2f}')

## Step 7 — Compare: Drift Report

In [ ]:
drift_report = compare(snapshot, result_drifted)

print(f'Has drift  : {drift_report.has_drift}')
print(f'Severity   : {drift_report.severity.value.upper()}')
print(f'Regressions: {len(drift_report.regressions)}')
print(f'Improvements: {len(drift_report.improvements)}')
print()
print(drift_report.summary)

In [ ]:
# Rich terminal drift report
DriftReporter(drift_report).print_terminal()

## Step 8 — Export Reports

In [ ]:
import json

drift_reporter = DriftReporter(drift_report)

# Markdown
md = drift_reporter.to_markdown()
print('=== Markdown Report (first 30 lines) ===')
print('\n'.join(md.split('\n')[:30]))
print('...')

In [ ]:
# JSON
js = drift_reporter.to_json()
parsed = json.loads(js)
print('=== JSON Keys ===')
print(list(parsed.keys()))
print(f'\nRegressions: {len(parsed["comparisons"])} comparisons total')

In [ ]:
# HTML (save and display path)
html_path = Path(tmp_dir) / 'drift_report.html'
drift_reporter.to_html(html_path)
print(f'✅ HTML report saved to: {html_path}')
print('   Open in your browser or use IPython.display.IFrame to preview.')

## Step 9 — Load from YAML

The recommended production workflow uses `canary.yaml` instead of Python construction.

In [ ]:
yaml_content = """
name: yaml-demo-suite
probes:
  - type: keyword_presence
    required_keywords: ["Paris"]
  - type: refusal
    expect_refusal: false
prompts:
  - text: "What is the capital of France?"
    id: geo
    expected_keywords: ["Paris"]
"""

yaml_path = Path(tmp_dir) / 'canary.yaml'
yaml_path.write_text(yaml_content)

yaml_suite = CanarySuite.from_yaml(yaml_path)
print(f'✅ Loaded suite from YAML: {yaml_suite.name}')
print(f'   Prompts: {len(yaml_suite.prompts)}, Probes: {len(yaml_suite.probes)}')

## Step 10 — Custom Probe with @probe Decorator

In [ ]:
from promptcanary.core.probes.base import probe
from promptcanary.core.models import CanaryPrompt, LLMResponse, ProbeCategory, ProbeResult

@probe("json_name_key", name="JSON Name Key Check", category=ProbeCategory.FORMAT)
def check_json_name(prompt: CanaryPrompt, response: LLMResponse) -> ProbeResult:
    """Verify the JSON response contains a 'name' key with a string value."""
    import json
    try:
        data = json.loads(response.content)
        has_name = isinstance(data.get('name'), str) and len(data['name']) > 0
    except (json.JSONDecodeError, AttributeError):
        has_name = False
    return ProbeResult(
        probe_id="json_name_key",
        probe_name="JSON Name Key Check",
        category=ProbeCategory.FORMAT,
        prompt_id=prompt.id,
        passed=has_name,
        score=1.0 if has_name else 0.0,
        details="'name' key present and non-empty." if has_name else "'name' key missing or empty.",
    )

# Use it:
custom_suite = CanarySuite(
    name="custom-probe-demo",
    prompts=[CanaryPrompt(id="json", text="Return JSON with a name key.")],
    probes=[check_json_name()],
)
result = custom_suite.run(MockProvider(drift=False), show_progress=False)
pr = result.probe_results[0]
print(f'Custom probe result: passed={pr.passed}, score={pr.score}, details="{pr.details}"')

---

## Summary

You've completed the full PromptCanary workflow:

| Step | Action | Result |
|------|--------|--------|
| 1 | Build suite | 3 prompts × 4 probes |
| 2 | Run (clean) | 100% pass rate |
| 3 | Save baseline | JSON snapshot on disk |
| 4 | Run (drifted) | Failures detected |
| 5 | Compare | Drift report with severity |
| 6 | Export | Markdown, HTML, JSON |

**Next steps:**
- Replace `MockProvider` with `LiteLLMProvider("openai/gpt-4o")`
- Add your own prompts that reflect real production patterns
- Add `promptcanary.yml` to your GitHub Actions for weekly scheduled checks

See [CONTRIBUTING.md](../CONTRIBUTING.md) to add a new probe or canary suite.